# PRISM: test everything

Two parts:

1. **Automated tests** (about 1 minute, no Gemini key needed): downloads the repository and runs the
   90 tests on made-up data. If they pass, the package works in Colab.
2. **A full run on your SomaScan file**: read → AI grouping → confirm → save → audit → plan → approve
   → verify → replay. To get through in one go, two helper functions **accept the AI's proposals and
   approve every Recommended step automatically**. That is for testing only; for real use, go through
   notebooks 05 and 10 and decide each step yourself.

At the end, a cell prints a short report. If anything fails, copy that report (and the error) back to me.

**Before you start:** Colab Secrets (key icon on the left) must hold `GITHUB_TOKEN` (with Notebook access);
`GEMINI_API_KEY` is optional (PRISM asks for the key once if it is not there). Runtime → Run all works.

## 0. Install

In [ ]:
import time, json, traceback
from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
BRANCH = "claude/prism-data-input-recognition-9adu1r"
!rm -rf /content/PRISM
!git clone -q -b {BRANCH} "https://{token}@github.com/Isra-inventor/PRISM.git" /content/PRISM
!pip install -q --force-reinstall --no-deps /content/PRISM
!pip install -q pytest
import prism
print("PRISM", prism.__version__)
REPORT = {"prism_version": prism.__version__, "steps": {}}

def note(step, ok, detail=""):
    REPORT["steps"][step] = {"ok": ok, "detail": str(detail)[:500]}
    print(("OK   " if ok else "FAIL ") + step + (f": {detail}" if detail else ""))

## Part 1. Automated tests

They use made-up data and a stand-in AI, so no key and no data are needed. Expect `90 passed`.

In [ ]:
t = time.time()
r = !cd /content/PRISM && python -m pytest -q 2>&1 | tail -3
print("\n".join(r))
note("pytest", any("passed" in x for x in r) and not any("failed" in x or "error" in x for x in r), r[-1] if r else "")
print(f"{time.time() - t:.0f} s")

## Part 2. A full run on your file

Change the settings below if your files or answers differ. `HISTORY` are your step-9 answers (processing before upload); `INTAKE` is what your study compares (`contrast_variable` must be a column of your sample information, or `"none"`).

In [ ]:
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/PRISM_data"
FILE = f"{DATA}/SomaExpr_common.csv"
SAMPLE_INFO = None                     # e.g. f"{DATA}/sample_info.csv" if you have one
OUT_DIR = "/content/drive/MyDrive/PRISM_out/test_run"
HISTORY = dict(normalized="yes", log_transformed="no", imputed="no", batch_corrected="no",
               features_or_samples_removed_before_upload="not_sure")
INTAKE = dict(contrast_variable="none", question_type="global_untargeted", downstream_family="unsure")

### Helpers (for this test run only)

In [ ]:
def quick_confirm(s, answers, time_unit="visit_index"):
    """FOR TESTING ONLY: accept every AI proposal so the whole pipeline can run once.
    Unresolved columns are set to 'ignore'. Prints everything it decided."""
    import prism
    todo = s.todo()
    print(f"{len(todo)} open item(s) before quick_confirm")
    for g in list(s.group_map.values()):
        if g.role == "unresolved":
            print(f"  {g.id} {g.columns[:3]} unresolved -> ignore")
            s.set(g.id, role="ignore", label="ignored in the test run")
    s.confirm_all()
    if s.layout != "samples_in_rows" and s.feature_id_decision["columns"]:
        s.confirm_feature_id()
    s.confirm_flags()
    for bid, g in s.block_map().items():
        x = s.matrix(bid, numbers=True)
        if g.zero_means is None and (x == 0).any().any():
            print(f"  {bid} has zeros -> treated as 'not_detected'")
            s.set_block(bid, zero_means="not_detected")
    s.confirm_values()
    try:
        s.propose_sample_types()
    except Exception as e:
        print("  sample types: AI call failed, every sample is a study sample:", e)
    s.confirm_samples()
    if s.info and not s.info["confirmed"]:
        s.confirm_sample_info()
    d = s.design()
    if d["time"]["source"] != "none" and not d.get("time_unit"):
        s.set_design(time_unit=time_unit)
    s.confirm_design()
    s.answer_history(**answers)
    left = s.todo()
    print(f"{len(left)} open item(s) after quick_confirm")
    return left


def approve_recommended(st, max_rounds=10):
    """FOR TESTING ONLY: approve the first Recommended candidate of each Recommended entry, in plan order,
    rebuilding the plan after each approval. Stops when nothing Recommended is left."""
    for _ in range(max_rounds):
        plan = st.build_plan()
        todo = [e for e in plan.entries if not e.get("checkpoint") and e["status"] == "Recommended"
                and any(c["status"] == "Recommended" for c in e["candidates"])
                and not any(d["entry_id"] == e["entry_id"] and d["decision"].startswith("approved")
                            for d in st.decisions)]
        if not todo:
            print("nothing Recommended left")
            return plan
        e = todo[0]
        i = next(k for k, c in enumerate(e["candidates"]) if c["status"] == "Recommended")
        print(f"approving {e['entry_id']}: {e['candidates'][i]['label']}")
        try:
            out = st.approve(e["entry_id"], i, user="test run")
            print("   ->", out["state"])
        except Exception as ex:
            print("   refused:", ex)
            st.reject(e["entry_id"], reason=f"refused in test run: {ex}")
    return st.build_plan()


### Step 1-3: read the file, and what the AI will see

In [ ]:
try:
    table = prism.read(FILE)
    display(table)
    d = prism.digest(table, prism.propose_layout(table))
    prism.show(d, max_chars=2500)
    note("read + digest", True, table.describe())
except Exception as e:
    note("read + digest", False, e); traceback.print_exc()

### Step 4: the AI groups the columns (Gemini)

Asks for the Gemini key here if it is not in Secrets.

In [ ]:
try:
    t = time.time()
    s = prism.start(table)
    display(s.groups())
    print("refused by code:", s.rejected)
    note("AI grouping", True, f"{len(s.group_map)} groups, layout {s.layout}, {len(s.rejected)} rejected, "
                              f"model {s.ai_models}, {time.time() - t:.0f} s")
except Exception as e:
    note("AI grouping", False, e); traceback.print_exc()

### Steps 5-10: accept the proposals (test only), then save

Look at the groups above first: anything the AI got wrong would normally be corrected here with `s.set(...)`, e.g. `s.set("g7", role="feature_annotation", label="dissociation constant")`.

In [ ]:
try:
    if SAMPLE_INFO:
        display(s.add_sample_info(SAMPLE_INFO))
    left = quick_confirm(s, HISTORY)
    display(left)
    s.review()
    s.save(OUT_DIR)
    note("schema confirmed + saved", len(left) == 0, f"{len(left)} open item(s); saved to {OUT_DIR}")
except Exception as e:
    note("schema confirmed + saved", False, e); traceback.print_exc()

### Step 11: the full audit (checkpoint C0)

In [ ]:
try:
    t = time.time()
    st = prism.Study(s, folder=f"{OUT_DIR}/work")
    summary = st.audit()
    display(summary)
    note("audit", True, dict(zip(summary["audit"], summary["status"])) | {"seconds": round(time.time() - t)})
except Exception as e:
    note("audit", False, e); traceback.print_exc()

In [ ]:
st.root_audit        # every finding with its sentences: read these

In [ ]:
display(st.questions())    # what only you can answer (processing-state mismatches, floors)
st.root_audit.findings["A13"]

### Step 12: intake and plan

In [ ]:
try:
    st.set_intake(**INTAKE)
    plan = st.build_plan()
    display(plan)
    note("plan", True, {e["entry_id"]: e["status"] for e in plan.entries if not e.get("checkpoint")})
except Exception as e:
    note("plan", False, e); traceback.print_exc()

### Approve every Recommended step (test only)

Each approval makes a new data state and re-runs the audit on it.

In [ ]:
try:
    t = time.time()
    plan = approve_recommended(st)
    display(plan)
    display(st.states())
    note("approve", True, f"{len(st.decisions)} decision(s), {len(st.store.chain())} op(s) applied, "
                          f"{time.time() - t:.0f} s")
except Exception as e:
    note("approve", False, e); traceback.print_exc()

### Step 13: verify (only after a batch correction or filtering step), replay, save

In [ ]:
try:
    if st.checkpoint_of(st.store.meta()) == "C2":
        display(st.verify())
        note("verify", True)
    else:
        display(st.compare())          # S0 against now, the same measures
        note("verify", True, "not at C2 (no batch correction / filtering approved): compared S0 with now instead")
    r = st.replay()
    note("replay", r["identical"], f"{len(r['steps'])} op(s) replayed")
    out = st.save()
    note("save study", True, sorted(p.name for p in out.iterdir()))
except Exception as e:
    note("verify / replay / save", False, e); traceback.print_exc()

## The report

Copy this back to me (with any error above).

In [ ]:
print(json.dumps(REPORT, indent=1, default=str))